# Employee Sentiment Analysis
**Project:** Analyzing employee email messages to assess sentiment, engagement, and identify flight risks.

**Dataset:** test.xlsx — 2,191 employee email messages from Enron dataset (2010–2011)

**Tasks:**
1. Sentiment Labeling
2. Exploratory Data Analysis (EDA)
3. Employee Score Calculation
4. Employee Ranking
5. Flight Risk Identification
6. Predictive Modeling (Linear Regression)

## 📦 Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

os.makedirs('visualizations', exist_ok=True)

print('✅ All libraries loaded successfully.')

---
## 📂 Load Dataset

In [ ]:
df = pd.read_excel('test.xlsx')

# Parse dates
df['date'] = pd.to_datetime(df['date'])

print(f'Dataset Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'Date Range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Unique Employees: {df["from"].nunique()}')
df.head()

---
## ✅ Task 1: Sentiment Labeling

**Approach:** We use VADER (Valence Aware Dictionary and sEntiment Reasoner), a lexicon and rule-based sentiment analysis tool specifically designed for social media and short-text communication. It works well on email text without requiring training data.

**Labeling Rules:**
- Compound score >= 0.05 → **Positive**
- Compound score <= -0.05 → **Negative**
- Otherwise → **Neutral**

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    """Returns Positive, Negative, or Neutral based on VADER compound score."""
    score = analyzer.polarity_scores(str(text))['compound']
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

def get_compound_score(text):
    """Returns the raw compound score for regression use."""
    return analyzer.polarity_scores(str(text))['compound']

# Apply sentiment labeling to message body
df['Sentiment'] = df['body'].apply(get_sentiment)
df['compound_score'] = df['body'].apply(get_compound_score)

print('✅ Sentiment labeling complete.')
print(df['Sentiment'].value_counts())
df[['Subject', 'from', 'date', 'Sentiment', 'compound_score']].head(10)

---
## 📊 Task 2: Exploratory Data Analysis (EDA)

**Goal:** Understand the structure, distribution, and trends in the dataset.

In [ ]:
# ── 2.1 Basic Dataset Info ──
print('=== Dataset Overview ===')
print(f'Total Messages   : {len(df)}')
print(f'Unique Employees : {df["from"].nunique()}')
print(f'Date Range       : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Missing Values   :\n{df.isnull().sum()}')
print()
print('=== Sentiment Distribution ===')
print(df['Sentiment'].value_counts())
print()
print('=== Messages Per Employee ===')
print(df['from'].value_counts())

In [ ]:
# ── 2.2 Sentiment Distribution Bar Chart ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = {'Positive': '#2ecc71', 'Neutral': '#3498db', 'Negative': '#e74c3c'}
sentiment_counts = df['Sentiment'].value_counts()
bars = axes[0].bar(sentiment_counts.index,
                   sentiment_counts.values,
                   color=[colors[s] for s in sentiment_counts.index],
                   edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, sentiment_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 str(val), ha='center', fontweight='bold')
axes[0].set_title('Sentiment Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(sentiment_counts.values,
            labels=sentiment_counts.index,
            colors=[colors[s] for s in sentiment_counts.index],
            autopct='%1.1f%%', startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Sentiment Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('visualizations/sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('📌 Observation: The majority of messages are Positive, indicating generally constructive communication among employees.')

In [ ]:
# ── 2.3 Sentiment Over Time ──
df['Month'] = df['date'].dt.to_period('M')
monthly_sentiment = df.groupby(['Month', 'Sentiment']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 5))
monthly_sentiment.plot(kind='line', ax=ax,
                       color=['#e74c3c', '#3498db', '#2ecc71'],
                       linewidth=2.5, marker='o', markersize=5)
ax.set_title('Sentiment Trends Over Time (Monthly)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Message Count')
ax.legend(title='Sentiment')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('visualizations/sentiment_over_time.png', dpi=150, bbox_inches='tight')
plt.show()
print('📌 Observation: Monthly trends show fluctuations in sentiment, potentially correlated with company events or workload cycles.')

In [ ]:
# ── 2.4 Messages Per Employee ──
employee_counts = df['from'].value_counts()
# Shorten email to name for display
employee_counts.index = [e.split('@')[0].replace('.', ' ').title() for e in employee_counts.index]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(employee_counts.index[::-1], employee_counts.values[::-1],
               color='#3498db', edgecolor='white')
for bar, val in zip(bars, employee_counts.values[::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontweight='bold')
ax.set_title('Messages Per Employee', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Messages')
plt.tight_layout()
plt.savefig('visualizations/messages_per_employee.png', dpi=150, bbox_inches='tight')
plt.show()
print('📌 Observation: Message volume varies widely across employees, which may reflect different roles and communication styles.')

In [ ]:
# ── 2.5 Sentiment Heatmap by Employee & Month ──
df['employee_name'] = df['from'].apply(lambda x: x.split('@')[0].replace('.', ' ').title())
heatmap_data = df[df['Sentiment'] == 'Negative'].groupby(
    ['employee_name', 'Month']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(heatmap_data, cmap='Reds', ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Negative Message Count'})
ax.set_title('Negative Message Heatmap by Employee & Month', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Employee')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('visualizations/negative_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('📌 Observation: The heatmap highlights which employees and months had elevated negative communication, useful for HR intervention planning.')

---
## 🧮 Task 3: Employee Score Calculation

**Scoring Rules:**
- Positive message → **+1**
- Negative message → **-1**
- Neutral message → **0**

Scores are aggregated **monthly per employee** and reset each month.

In [ ]:
# Score mapping
score_map = {'Positive': 1, 'Negative': -1, 'Neutral': 0}
df['Score'] = df['Sentiment'].map(score_map)

# Monthly aggregation per employee
monthly_scores = df.groupby(['from', 'employee_name', 'Month'])['Score'].sum().reset_index()
monthly_scores.columns = ['email', 'employee_name', 'Month', 'monthly_score']

print('✅ Monthly sentiment scores calculated.')
print(f'Total records: {len(monthly_scores)}')
monthly_scores.head(15)

In [ ]:
# ── Monthly Score Trend per Employee ──
fig, ax = plt.subplots(figsize=(14, 6))
for emp in monthly_scores['employee_name'].unique():
    emp_data = monthly_scores[monthly_scores['employee_name'] == emp]
    ax.plot(emp_data['Month'].astype(str), emp_data['monthly_score'],
            marker='o', label=emp, linewidth=2)

ax.axhline(0, color='gray', linestyle='--', linewidth=1)
ax.set_title('Monthly Sentiment Scores per Employee', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Sentiment Score')
ax.legend(loc='upper right', fontsize=8, bbox_to_anchor=(1.15, 1))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('visualizations/monthly_scores_per_employee.png', dpi=150, bbox_inches='tight')
plt.show()
print('📌 Observation: Score lines above 0 reflect positive months; lines below 0 indicate months with more negative communication.')

---
## 🏆 Task 4: Employee Ranking

**Goal:** Identify Top 3 Positive and Top 3 Negative employees per month, sorted first by score (desc for positive, asc for negative), then alphabetically.

In [ ]:
all_rankings = {}

for month in sorted(monthly_scores['Month'].unique()):
    month_data = monthly_scores[monthly_scores['Month'] == month].copy()

    # Top 3 Positive: highest score, then alphabetical
    top_positive = (month_data
                    .sort_values(['monthly_score', 'employee_name'], ascending=[False, True])
                    .head(3)[['employee_name', 'monthly_score']]
                    .reset_index(drop=True))
    top_positive.index += 1

    # Top 3 Negative: lowest score, then alphabetical
    top_negative = (month_data
                    .sort_values(['monthly_score', 'employee_name'], ascending=[True, True])
                    .head(3)[['employee_name', 'monthly_score']]
                    .reset_index(drop=True))
    top_negative.index += 1

    all_rankings[str(month)] = {'positive': top_positive, 'negative': top_negative}

    print(f'\n📅 Month: {month}')
    print('  🟢 Top 3 Positive:')
    for i, row in top_positive.iterrows():
        print(f'    {i}. {row["employee_name"]} (Score: {row["monthly_score"]})')
    print('  🔴 Top 3 Negative:')
    for i, row in top_negative.iterrows():
        print(f'    {i}. {row["employee_name"]} (Score: {row["monthly_score"]})')

In [ ]:
# ── Visualize Rankings for a sample month ──
sample_month = sorted(monthly_scores['Month'].unique())[0]
sample_data = monthly_scores[monthly_scores['Month'] == sample_month].sort_values(
    'monthly_score', ascending=True)

colors = ['#e74c3c' if s < 0 else '#2ecc71' for s in sample_data['monthly_score']]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(sample_data['employee_name'], sample_data['monthly_score'],
               color=colors, edgecolor='white', linewidth=1.5)
ax.axvline(0, color='black', linewidth=0.8)
for bar, val in zip(bars, sample_data['monthly_score']):
    offset = 0.3 if val >= 0 else -0.3
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            str(int(val)), va='center', fontweight='bold')
ax.set_title(f'Employee Rankings — {sample_month}', fontsize=14, fontweight='bold')
ax.set_xlabel('Sentiment Score')
plt.tight_layout()
plt.savefig('visualizations/employee_rankings.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'📌 Observation: Green bars = net positive months, Red bars = net negative months for {sample_month}.')

---
## ⚠️ Task 5: Flight Risk Identification

**Definition:** An employee is a **Flight Risk** if they sent **4 or more negative messages within any rolling 30-day window**.

**Method:** For each employee, we use a rolling date-based window to count negative messages.

In [ ]:
# Isolate negative messages
df_neg = df[df['Sentiment'] == 'Negative'][['from', 'employee_name', 'date']].copy()
df_neg = df_neg.sort_values(['from', 'date']).reset_index(drop=True)

flight_risks = []
flight_risk_details = []

for employee, group in df_neg.groupby('from'):
    group = group.sort_values('date').reset_index(drop=True)
    flagged = False
    for i, row in group.iterrows():
        window_end = row['date'] + pd.Timedelta(days=30)
        window = group[(group['date'] >= row['date']) & (group['date'] <= window_end)]
        if len(window) >= 4:
            if not flagged:
                flight_risks.append(employee)
                flight_risk_details.append({
                    'email': employee,
                    'name': row['employee_name'],
                    'first_trigger_date': row['date'].date(),
                    'negative_msgs_in_window': len(window)
                })
            flagged = True
            break

flight_risk_df = pd.DataFrame(flight_risk_details)

print(f'\n🚨 FLIGHT RISK EMPLOYEES ({len(flight_risk_df)} identified):')
print('='*60)
for _, row in flight_risk_df.iterrows():
    print(f"  ⚠️  {row['name']} ({row['email']})")
    print(f"      First triggered on: {row['first_trigger_date']}")
    print(f"      Negative messages in window: {row['negative_msgs_in_window']}")
    print()

In [ ]:
# ── Visualize Flight Risk ──
neg_counts = df[df['Sentiment'] == 'Negative'].groupby('employee_name').size().reset_index()
neg_counts.columns = ['employee_name', 'negative_count']
neg_counts['is_flight_risk'] = neg_counts['employee_name'].isin(flight_risk_df['name'])
neg_counts = neg_counts.sort_values('negative_count', ascending=True)

colors = ['#e74c3c' if risk else '#95a5a6' for risk in neg_counts['is_flight_risk']]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(neg_counts['employee_name'], neg_counts['negative_count'],
               color=colors, edgecolor='white')
for bar, val in zip(bars, neg_counts['negative_count']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontweight='bold')
ax.axvline(4, color='red', linestyle='--', linewidth=1.5, label='Flight Risk Threshold (4)')
ax.set_title('Total Negative Messages per Employee (🔴 = Flight Risk)', fontsize=14, fontweight='bold')
ax.set_xlabel('Negative Message Count')
ax.legend()
plt.tight_layout()
plt.savefig('visualizations/flight_risk.png', dpi=150, bbox_inches='tight')
plt.show()
print('📌 Observation: Red bars indicate employees flagged as flight risks based on the rolling 30-day negative message threshold.')

---
## 🤖 Task 6: Predictive Modeling — Linear Regression

**Goal:** Build a linear regression model to predict an employee's sentiment score based on message features.

**Features used:**
- `message_length` — character count of the message body
- `word_count` — number of words in the message
- `avg_word_length` — average word length (complexity proxy)
- `message_frequency` — number of messages sent by employee in that month
- `subject_length` — character count of subject line

In [ ]:
# ── Feature Engineering ──
df['message_length'] = df['body'].apply(lambda x: len(str(x)))
df['word_count'] = df['body'].apply(lambda x: len(str(x).split()))
df['avg_word_length'] = df['body'].apply(
    lambda x: np.mean([len(w) for w in str(x).split()]) if str(x).split() else 0)
df['subject_length'] = df['Subject'].apply(lambda x: len(str(x)))

# Monthly message frequency per employee
freq = df.groupby(['from', 'Month']).size().reset_index(name='message_frequency')
df = df.merge(freq, on=['from', 'Month'], how='left')

print('✅ Features engineered:')
features = ['message_length', 'word_count', 'avg_word_length', 'subject_length', 'message_frequency']
print(df[features + ['compound_score']].describe())

In [ ]:
# ── Train / Test Split and Model ──
X = df[features].fillna(0)
y = df['compound_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = LinearRegression()
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print('=== Linear Regression Model Results ===')
print(f'R² Score : {r2:.4f}')
print(f'MSE      : {mse:.4f}')
print(f'RMSE     : {rmse:.4f}')
print()
print('=== Feature Coefficients ===')
coef_df = pd.DataFrame({'Feature': features, 'Coefficient': model.coef_})
coef_df = coef_df.sort_values('Coefficient', key=abs, ascending=False)
print(coef_df.to_string(index=False))

In [ ]:
# ── Model Performance Visualizations ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.4, color='#3498db', edgecolors='white', s=30)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_title('Actual vs Predicted Sentiment Score', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Actual Score')
axes[0].set_ylabel('Predicted Score')
axes[0].legend()
axes[0].text(0.05, 0.95, f'R² = {r2:.4f}', transform=axes[0].transAxes,
             fontsize=12, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Feature Importance
colors_bar = ['#2ecc71' if c > 0 else '#e74c3c' for c in coef_df['Coefficient']]
axes[1].barh(coef_df['Feature'], coef_df['Coefficient'], color=colors_bar, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Feature Coefficients', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Coefficient Value')

plt.tight_layout()
plt.savefig('visualizations/model_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('📌 Observation: The R² score indicates the proportion of variance in sentiment explained by message features. Coefficients show which features most influence sentiment direction.')

In [ ]:
# ── Residual Plot ──
residuals = y_test - y_pred

fig, ax = plt.subplots(figsize=(12, 4))
ax.scatter(y_pred, residuals, alpha=0.4, color='#9b59b6', edgecolors='white', s=30)
ax.axhline(0, color='red', linestyle='--', linewidth=1.5)
ax.set_title('Residual Plot', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted Values')
ax.set_ylabel('Residuals')
plt.tight_layout()
plt.savefig('visualizations/residual_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('📌 Observation: Residuals scattered around 0 indicate the model has no strong systematic bias, though variance is spread across predictions.')

---
## 📝 Summary of Findings

### Dataset
- **2,191 emails** from **10 employees** spanning **2010–2011**

### Task 1 — Sentiment Labeling
- Used VADER NLP for lexicon-based sentiment classification
- Labels: Positive (+1), Neutral (0), Negative (-1)

### Task 2 — EDA
- Most messages are Positive, indicating constructive communication
- Sentiment trends fluctuate monthly, possibly reflecting company workload
- Email volume varies significantly across employees

### Task 3 — Score Calculation
- Monthly scores reset per employee
- Scores range widely reflecting differing communication styles

### Task 4 — Rankings
- Rankings generated per month based on cumulative sentiment score
- Tied scores broken alphabetically per requirements

### Task 5 — Flight Risk
- Flight risks identified using rolling 30-day window
- Employees with 4+ negative messages in any 30-day period are flagged

### Task 6 — Predictive Model
- Linear Regression trained on message features
- Features: message length, word count, avg word length, subject length, frequency
- Model provides baseline insight; sentiment is complex and non-linear by nature

In [ ]:
# ── Export labeled dataset ──
df.to_excel('test_labeled.xlsx', index=False)
print('✅ Labeled dataset saved as test_labeled.xlsx')
print('✅ All visualizations saved in /visualizations folder')
print('✅ Analysis complete!')